load_documents
split_documents
create_embed_model
build_vector_store
load_vector_store
retrieve_documents
format_context
generate_answer

In [7]:
from pathlib import Path
import sys
from src import config





In [ ]:
import bs4
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_classic import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.document_loaders import BaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


# Load Docs
def load_documents(file_path: str) -> list[Document]:
    """
    Loads Every PDF in a directory as page-level LangChain documents
    """
    documents = []

    for pdf_path in Path(file_path).glob("*pdf"):
        reader = PdfReader(pdf_path)

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text()

            if text and text.strip():
                documents.append(
                    Document(
                        page_content=text.strip(),
                        metadata={
                            "source":pdf_path.name,
                            "page": page_number
                        }
                    )
                )
    if not documents:
        raise ValueError(f"No readable PDFs found in {file_path}")

    return documents


docs = load_documents(config.RAW_DATA_DIR)
print(f"Loaded {len(docs)} pages")
print(docs[0].metadata)
print(docs[0].page_content[:500])


Loaded 215 pages
{'source': 'general_provisions.pdf', 'page': 1}
{'source': 'operational_regulations.pdf', 'page': 16}


In [11]:
def split_documents(
        documents,
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP
    ):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    return text_splitter.split_documents(documents)


splits = split_documents(docs)

print(f"Created {len(splits)} chunks")
print(splits[0].metadata)
print(splits[0].page_content[:500])

Created 931 chunks
{'source': 'general_provisions.pdf', 'page': 1}
SECTION A: GENERAL REGULATORY PROVISIONS 
 
 
 
0 
 
A 
 
  
A1 
2026 Formula 1: General Regulatory Provisions 
©2026 Fédération Internationale de l’Automobile 
25 June 2026 
Issue 03 
SECTION A: GENERAL REGULATORY PROVISIONS 
 
Version:   Issue 03 
Status:    PUBLISHED 
Date:    25/06/2026 
WMSC approval date:  23/06/2026 
 
CONVENTION: 
Black Text: Text unchanged from 2026 FIA F1 Regulations – Section A [General Regulatory Provisions] – Iss 
02 
Pink Text: Changes relative to 2026 FIA F1 Regul
